# Homework 1

#### EE-556 Mathematics of Data - Fall 2026


In this homework, we consider a multiclass classification task modeled by multinomial (softmax) logistic regression. Your goal will be to analyze the estimator and its properties (convexity, existence/uniqueness), and to derive gradients/Hessians and smoothness bounds. The first part consists of theoretical questions only.


<div class="alert alert-info">
  ℹ️ <strong>Information on group based work:</strong>
</div>

- Only one group member needs to submit all five completed notebooks on behalf of the group.
- You may seek help outside your group, but each group must write its own answers.
- If the work was <span style="color: red;">unevenly distributed</span> among group members, let the TAs know.
- Only one member of the group is allowed to use AI. We will require sharing the conversation history with the AI in the form of a public link. If you use multiple conversations across the same or multiple tools please share all of them. Name the person in your group who is allowed to use AI. We encourage you to use the AI to help you understand the material, but we ask you to write the code and theory solutions by yourself. Part 5 permits AI usage as specified in its instructions.

<div style="border: 1px solid #f00; background-color: #fdd; padding: 10px; border-radius: 5px;">
  ⚠️ Do not forget: List each group member's name and SCIPER number.
</div>


Person 1 **Francesco Foglia**: || Person 1 **425511**: 


Person 2 **Antonio Angelo Polignano**: || Person 2 **436345**:


Person 3 **Name**: || Person 3 **SCIPER**:


<div style="border: 1px solid #0a0; background-color: #dfd; padding: 10px; border-radius: 5px;">
  📓 Feedback on AI use: Please use the following cell to provide feedback on the AI use in this notebook.
  
  For example, how useful were the tools to you? Which tools did you use? Did you feel like they helped you understand the material better?
</div>

*Describe your use of AI tools here.*


## 1. Multiclass Softmax Logistic Regression - 30 Points


We now model multiclass classification with classes $c \in \{1,\dots,C\}$. For each sample $(\mathbf{a}_i, b_i)$ with $\mathbf{a}_i \in \mathbb{R}^p$ and $b_i \in \{1,\dots,C\}$, let $\mathbf{X} = [\mathbf{x}_1,\dots,\mathbf{x}_C] \in \mathbb{R}^{p\times C}$ be the class weight matrix. The softmax model defines

$$
\mathbb{P}(b_i = c \mid \mathbf{a}_i) = \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_c)}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)}.
$$

Assume i.i.d. samples $\{(\mathbf{a}_i,b_i)\}_{i=1}^n$. Our goal is to estimate $\mathbf{X}$ by maximum likelihood (and later with an $\ell_2$ regularizer).


__(a)__ (2 points) Show that the negative log-likelihood $f$ can be written as:

$$
\begin{aligned}
 f(\mathbf{X})
 &= - \log \mathbb{P}(b_1,\dots,b_n\mid \mathbf{a}_1,\dots,\mathbf{a}_n)\\
 &= \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right].
\end{aligned}
$$



By i.i.d, the joint law can be computed as
$$
\begin{align*}
f(\mathbf{X}) 
&= - \log \prod_{i=1}^n \mathbb{P}(b_i \mid \mathbf{a_i}) \\
&= - \sum _{i=1}^n \log \mathbb{P}(b_i \mid \mathbf{a_i}) 
\end{align*}
$$
We now look at each individual term
$$
\begin{align*}
\log \mathbb{P}(b \mid \mathbf{a}^\top) 
&= \log \left( \frac{\exp(\mathbf{a}^\top \mathbf{x}_{b_i})}{\sum_{k=1}^C \exp(\mathbf{a}^\top \mathbf{x}_k)}\right) \\
&= \log(\exp(\mathbf{a}^\top \mathbf{x}_{b_i})) - \log \left(\sum_{k=1}^C \exp(\mathbf{a}^\top \mathbf{x}_k)\right) \\
&= \mathbf{a}^\top \mathbf{x}_{b_i} - \log \left(\sum_{k=1}^C \exp(\mathbf{a}^\top \mathbf{x}_k)\right)
\end{align*}
$$
by substituting in the first equality obtain the requested identity.

__(b)__ (3 points) Show that $\mathbf{u} \mapsto \log\!\left(\sum_{k=1}^C e^{u_k}\right)$ is convex on $\mathbb{R}^C$. Then, show that $f(\mathbf{X})$ is convex.


Hint: use Jensen's inequality.

We prove convexity by considering the standard algebraic definition. Let $g$ be the map in definition and $\lambda \in (0, 1)$
$$
\begin{align*}
g(\lambda \mathbf{x} + (1 - \lambda) \mathbf{y})
&= \log(\sum_{k=1}^C e^{\lambda x_k + (1 - \lambda) y_k}) \\
&= \log(\sum_{k=1}^C e^{\lambda x_k} e^{(1 - \lambda) y_k}) \\
&\leq \log \left(\sum_{k=1}^C e^{x_k} \right)^\lambda \left(\sum_{k=1}^C e^{y_k} \right)^{1 - \lambda} \\
&= \log \left(\sum_{k=1}^C e^{x_k} \right)^\lambda + \log \left(\sum_{k=1}^C e^{y_k} \right)^{1 - \lambda} \\
&= \lambda \log\left(\sum_{k=1}^C e^{x_k} \right) + (1 - \lambda) \log \left(\sum_{k=1}^C e^{y_k} \right) \\
&= \lambda g(\mathbf{x}) + (1 - \lambda) g(\mathbf{y}) \\ \\
\implies g(\lambda \mathbf{x} + (1 - \lambda) \mathbf{y}) &\leq \lambda g(\mathbf{x}) + (1 - \lambda) g(\mathbf{y})
\end{align*}
$$
where the inequality step is using Holder's inequality with exponents $p = \frac{1}{\lambda}, q = \frac{1}{1 - \lambda}$ 
and the monotonicity of the logarithm.

Let $h(\mathbf{x}) = \mathbf{a}^\top \mathbf{x}$. The function $f(\mathbf{X}) = \left(-h + (g \circ h)\right)(\mathbf{X})$ is itself convex as it's the sum of an affine function $h$ and the composition of a convex function $g$ with an affine function $h$.

Since the negative log-likelihood $f$ is convex, every local minimizer is a global minimizer. The maximum likelihood estimation problem is
$$
\mathbf{X}^\star_{ML} = \arg\min_{\mathbf{X} \in \mathbb{R}^{p\times C}} f(\mathbf{X})
$$

But does $f$ always attain its infimum? The following three questions examine this issue.


__(c)__ (1 point) Explain the difference between infima and minima. Give an example of a convex function on $\mathbb{R}$ that does not attain its infimum.


The infimum of a set is always well defined on the extended real numbers and it is equal to the minimum whenever the latter is well defined. In terms of functions, the infimum is the largest value that is smaller than or equal to any value in the image, but the infimum might not be part of the image, or it could not be $-\infty$. Whenever the infimum is finite and it is attained (i.e. it is part of the image), then the minimum is well defined and is the infimum itself.


The function $f(x) = e^x$ has the requested properties. Its infimum is $0$, it is convex, but there is no finite real value such that $f(x) = 0$. 

__(d)__ (2 points) Assume there exists $\mathbf{X}_0 \in \mathbb{R}^{p\times C}$ such that for all $i$,
$$
\mathbf{a}_i^\top \mathbf{x}_{0, b_i} - \max_{k \neq b_i} \mathbf{a}_i^\top \mathbf{x}_{0,k} > 0.
$$
This is called one-versus-all complete separation in multiclass settings. Give a geometric interpretation (e.g., for $p=2$) and explain why the name is appropriate.


It's helpful to look at the equations by grouping samples of the same class and factoring. 
$$
\begin{cases}
\mathbf{a}_{c,1}^\top (\mathbf{x}_{0, c} - \mathbf{x}_{0,k}) > 0 \quad \text{for all } k \neq c\\
\mathbf{a}_{c,2}^\top (\mathbf{x}_{0, c} - \mathbf{x}_{0,k}) > 0 \quad \text{for all } k \neq c\\
... \\
\end{cases}
$$ 
In geometrical terms, these equations are describing $C - 1$ hyperplanes (passing through the origin) that have all points of class $c$ on the same side (respectively the hyperplane passing through the origin that is orthogonal to $(\mathbf{x}_{0, c} - \mathbf{x}_{0,k})$). For samples of a fixed class $k \neq c$, we can look at the same equation and obtain
$$
\begin{cases}
\mathbf{a}_{k,1}^\top (\mathbf{x}_{0, k} - \mathbf{x}_{0,c}) > 0 \\
\mathbf{a}_{k,2}^\top (\mathbf{x}_{0, k} - \mathbf{x}_{0,c}) > 0 \\
... \\
\end{cases}
$$  
So the hyperplane defined by $(\mathbf{x}_{0, c} - \mathbf{x}_{0,k})$ is strictly separating classes $c$ and $k$. We now have pairwise separation, to get a "one-versus-all" separation we consider the $C-1$ $p$-dimensional regions of space that separate class $c$ from each of the other classes. If we intersect all such regions, we obtain a region of space (still $p$-dimensional) that separates class $c$ from all other classes!

Since the derivation is independent of the chosen class $c$ and the separations are defined by the previous hyperplanes, we can see that the inequalities are really defining a partitioning of space in $C$ $p$-dimensional regions, so that samples of the same class are in the same region. 

Complete separation allows perfect classification of the training data. However, as you will show next, it prevents the existence of a finite maximum likelihood estimate.


__(e)__ (3 points) In a one-versus-all complete separation setting (as in (d)), prove that $f$ does not attain its minimum. Hint: consider $f(\alpha \mathbf{X}_0)$ as $\alpha \to +\infty$ and compare it to $f(\mathbf{X}_0)$.


We first prove that the function $f$ is strictly positive for any $\mathbf{X}$.
$$
\begin{align*}
f(\mathbf{X}) 
&= \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right] \\
&> \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \log \exp(\mathbf{a}_i^\top \mathbf{x}_{b_i}) \right] \\
&= \sum_{i=1}^n \left[ -\mathbf{a}_i^\top \mathbf{x}_{b_i} + \mathbf{a}_i^\top \mathbf{x}_{b_i} \right] \\
\implies f(\mathbf{X}) &> 0 
\end{align*}
$$
where the inequality step is justified by positivity of the exponential function.

We now aim to prove that the infimum is $0$, by finding a sequence of values that has limit in $0$, because we know that it cannot be attained.
$$
\begin{align*}
f(\alpha X_0) 
&= \sum_{i=1}^n \left[ -\alpha \mathbf{a}_i^\top \mathbf{x}_{0, b_i} + \log \sum_{k=1}^C \exp(\alpha \mathbf{a}_i^\top \mathbf{x}_{0, k}) \right] \\
&= \sum_{i=1}^n \left[ \log \exp (-\alpha \mathbf{a}_i^\top \mathbf{x}_{0, b_i}) + \log \sum_{k=1}^C \exp(\alpha \mathbf{a}_i^\top \mathbf{x}_{0, k}) \right] \\
&= \sum_{i=1}^n \left[ \log \left( \exp (-\alpha \mathbf{a}_i^\top \mathbf{x}_{0, b_i}) \left(\sum_{k=1}^C \exp(\alpha \mathbf{a}_i^\top \mathbf{x}_{0, k}) \right)\right) \right] \\
&= \sum_{i=1}^n \left[ \log \left( \exp \left(\sum_{k=1}^C -\alpha \mathbf{a}_i^\top \mathbf{x}_{0, b_i} + \alpha \mathbf{a}_i^\top \mathbf{x}_{0, k} \right) \right) \right] \\
&= \sum_{i=1}^n \left[ \log \left( \exp \left(\alpha \sum_{k=1}^C \mathbf{a}_i^\top (-\mathbf{x}_{0, b_i} + \mathbf{x}_{0, k}) \right) \right) \right] \\
&= \sum_{i=1}^n \left[ \log (1 + \exp (\alpha s_i)) \right] \quad \text{where } s_i := \sum_{k \neq b_i} \mathbf{a}_i^\top (-\mathbf{x}_{0, b_i} + \mathbf{x}_{0, k}) < 0\\
\end{align*}
$$
where the negativity of $s_i$ is implied by the one-above-all separation bound.


By continuity of the logarithm and the exponential and $\lim_{\alpha \to \infty} \exp(\alpha s_i) = 0$ (as we have $s_i < 0) we can conclude
$$
\lim_{\alpha \to \infty} f(\alpha X_0) = 0 
$$
Therefore the infimum of $f$ is 0 but it cannot be finitely attained as a minimum.

We resolve this issue by adding a regularizer. Consider the regularized function

$$
 f_\mu(\mathbf{X}) = f(\mathbf{X}) + \frac{\mu}{2} \|\mathbf{X}\|_F^2, \quad \mu > 0.
$$

__(f)__ (2 points) Show that the gradient with respect to $\mathbf{X}$ of $f_\mu$ can be expressed as
$$
 \nabla_{\mathbf{X}} f_\mu(\mathbf{X}) = \sum_{i=1}^n \mathbf{a}_i \big( \mathbf{p}_i - \mathbf{e}_{b_i} \big)^\top + \mu \mathbf{X},
$$
where $\mathbf{e}_{b_i} \in \mathbb{R}^C$ is the [one-hot vector](https://en.wikipedia.org/wiki/One-hot) for class $b_i$, $\mathbf{p}_i \in \mathbb{R}^C$ has entries $p_{i,c} = \mathbb{P}(b_i=c\mid \mathbf{a}_i)$ under the softmax model, and $\mathbf{a}_i(\mathbf{p}_i - \mathbf{e}_{b_i})^\top\in\mathbb{R}^{p\times C}$ denotes the outer product.



We study the following three terms separately
$$ 
\sum_{i=1}^n \mathbf{a}_i \mathbf{p}_i^\top, \quad -\sum_{i=1}^n \mathbf{a}_i \mathbf{e}_{b_i}^\top, \quad \mu \mathbf{X}
$$

We can easily see that the last term is the gradient of the term with the frobenius norm (as it behaves as the standard vector 2-norm).

The second term is the gradient of $\sum_{i=1}^n -\mathbf{a}_i^\top \mathbf{x}_{b_i}$, as the tensor product with one-hot vector is precisely summing vector $\mathbf{a_i}$ in the $b_i$-th column of the gradient of $f_\mu$.

The only remaining term is the one originating from the logarithm in $f_\mu$, we compute the gradient with respect to $\mathbf{X}$ column-by-column. For a fixed sample $i$ and column $j \in \{1, \dots, C\}$:
$$
\begin{align*}
\nabla_{\mathbf{x}_j} \left( \log \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right)
&= \frac{1}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)} \nabla_{\mathbf{x}_j} \left( \sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k) \right) \\
&= \frac{\exp(\mathbf{a}_i^\top \mathbf{x}_j)}{\sum_{k=1}^C \exp(\mathbf{a}_i^\top \mathbf{x}_k)} \mathbf{a}_i \\
&= (\mathbf{p}_i)_j \mathbf{a}_i
\end{align*}
$$

Arranging these columns for $j = 1, \dots, C$ into a matrix gives the outer product for sample $i$:
$$
\left[ (\mathbf{p}_i)_1 \mathbf{a}_i, \; (\mathbf{p}_i)_2 \mathbf{a}_i, \; \dots, \; (\mathbf{p}_i)_C \mathbf{a}_i \right] = \mathbf{a}_i \mathbf{p}_i^\top
$$

Summing over all $n$ samples yields the desired first term:
$$
\sum_{i=1}^n \mathbf{a}_i \mathbf{p}_i^\top
$$

__(g)__ (4 points) Use row-wise vectorization of $\mathbf{X}$, obtained by stacking its rows one after another into a vector in $\mathbb{R}^{pC}$. Show that the Hessian of $f_\mu$ in these coordinates can be written as
<a id="eq-softmax-hessian"></a>

$$
 \nabla^2 f_\mu(\mathbf{X}) = \sum_{i=1}^n (\mathbf{a}_i\mathbf{a}_i^\top) \otimes \big( \operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top \big) + \mu \mathbf{I}_{pC},\tag{1}
$$
where $\otimes$ is the Kronecker product, and $\operatorname{Diag}(\mathbf{p}_i) - \mathbf{p}_i\mathbf{p}_i^\top$ is the softmax Jacobian, which is positive semidefinite.


*Write your answer here.*


__(h)__ (2 points) Show that $f_\mu$ is $\mu$-strongly convex.


*Write your answer here.*


__(i)__ (2 points) Is it possible for a strongly convex function to not attain its minimum? Justify your reasoning (you may assume the domain is $\mathbb{R}^{p\times C}$).


*Write your answer here.*


We will now show that $f_\mu$ is smooth, i.e., $\nabla f_\mu$ is L-Lipschitz with respect to the Frobenius norm, with a simple conservative bound
$$
 L = \|\mathbf{A}\|_F^2 + \mu.
$$
where
$$
 \mathbf{A} = \begin{bmatrix}
  \leftarrow &  \mathbf{a}_1^\top & \rightarrow \\
  \leftarrow &  \mathbf{a}_2^\top & \rightarrow \\
   &  \ldots &  \\
  \leftarrow &  \mathbf{a}_n^\top & \rightarrow \\
 \end{bmatrix}.
$$
(You may use that the operator norm of the softmax Jacobian is bounded by $1/2$, and a looser bound $\le 1$ is acceptable for grading.)

Hint: check the properties of the spectral norm with respect to dot product, Kronecker product, and outer product.

(4 points for all three questions)


__(j-1)__ Show that $\lambda_{\max}(\mathbf{a}_i\mathbf{a}_i^T) = \left\| \mathbf{a}_i\right\|_2^2$, where $\lambda_{\max}(\cdot)$ denotes the largest eigenvalue.


*Write your answer here.*


__(j-2)__ Using [(1)](#eq-softmax-hessian), show that $\lambda_{\max}(\nabla^2 f_\mu(\mathbf{X})) \leq \sum_{i=1}^{n} \|\mathbf{a}_i\|_2^2 + \mu$.

*Write your answer here.*


__(j-3)__ Conclude that $f_\mu$ is $L$-smooth for $L = \|\mathbf{A}\|_F^2 + \mu$.


*Write your answer here.*


__(l)__ (3 points) KL divergence and NLL. Let $q_i=(q(c\mid\mathbf{a}_i))_{c=1}^C$ be the true conditional label distribution and $\mathbf{p}_i$ the model softmax distribution. Write $\mathrm{KL}(q_i\,\|\,\mathbf{p}_i)$ and show that minimizing its average over the samples is equivalent to minimizing the expected average negative log-likelihood, where the expectation is over labels drawn from $q$ with the features held fixed. Explain how $f(\mathbf{X})/n$ from (a) estimates this expected loss.

*Write your answer here.*


The unregularized maximum likelihood problem may have no finite minimizer. Adding a quadratic regularizer with $\mu>0$ gives a unique regularized estimate, defined by the smooth, strongly convex problem:
<a id="eq-regularized-estimator"></a>

$$
 \mathbf{X}^\star = \arg\min_{\mathbf{X} \in \mathbb{R}^{p\times C}} f(\mathbf{X}) + \frac{\mu}{2}\|\mathbf{X}\|_F^2.\tag{2}
$$


## Binary logistic regression (specialization for Part 2)

While this part analyzed the multiclass (softmax) setting, in the next exercise we will continue under the simplified two-class case.

Let labels be $b_i \in \{-1, +1\}$, features $\mathbf{a}_i \in \mathbb{R}^p$, and weight vector $\mathbf{x} \in \mathbb{R}^p$. Define the sigmoid
$$
\sigma(t) = \frac{1}{1+e^{-t}}.
$$
Model the conditional distribution as
$$
\mathbb{P}(b_i = j \mid \mathbf{a}_i) = \sigma\big(j\, \mathbf{a}_i^\top \mathbf{x}\big), \quad j \in \{-1,+1\}.
$$
The likelihood over i.i.d. samples $\{(\mathbf{a}_i, b_i)\}_{i=1}^n$ is
$$
\mathcal{L}(\mathbf{x}) = \prod_{i=1}^n \sigma\big(b_i\, \mathbf{a}_i^\top \mathbf{x}\big),
$$
so the negative log-likelihood is
$$
 f(\mathbf{x}) = -\log \mathcal{L}(\mathbf{x}) = \sum_{i=1}^n \log\big(1 + e^{-b_i\, \mathbf{a}_i^\top \mathbf{x}}\big).
$$

__(m)__ (2 points) Show that the gradient of the negative log-likelihood is the standard binary logistic regression gradient:
$$
\nabla f(\mathbf{x}) = \sum_{i=1}^n \big(-b_i\, \sigma(-b_i\, \mathbf{a}_i^\top \mathbf{x})\big)\, \mathbf{a}_i.
$$
(Hint: use the chain rule and $\sigma'(t) = \sigma(t)\big(1-\sigma(t)\big)$.)

We will use this binary formulation in Part 2 - First order methods.


*Write your answer here.*
